# Étteremértékelések elemzése – 06. notebook
## Personalized Recommendation természetes nyelvű preferenciák alapján

Ebben a notebookban elkészítjük a projekt végső alkalmazási rétegét.

**Bemenet:** `place_profiles_wide.csv`, benne a supporttal korrigált (`*_adjusted`) helyprofilok.

A rendszer:
1. hibrid módon értelmezi a természetes nyelvű preferenciát (explicit szabályok + szemantikus fallback);
2. külön kezeli a venue type-ot és a „not important” típusú negációkat;
3. preference-súlyokat képez;
4. a súlyokat összeveti a place profile dimenzióival;
5. Top-N ajánlást és score-alapú magyarázatot ad.

**Fontos:** a `Price` dimenzió price/value satisfactiont mér, nem objektív olcsóságot.

## 1. Környezet

A preference parsinghez a `sentence-transformers/all-MiniLM-L6-v2` modellt használjuk. A place profile-okhoz már nem szükséges GPU, de az embedding modell GPU-val gyorsabban fut.


In [ ]:
!pip -q install sentence-transformers pandas numpy matplotlib

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 50)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


## 2. Place profile feltöltése

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    print("Nem Google Colab környezet.")


In [ ]:
PROFILE_PATH = Path("place_profiles_wide.csv")
assert PROFILE_PATH.exists(), f"Hiányzó fájl: {PROFILE_PATH}"

profiles_df = pd.read_csv(PROFILE_PATH)

print("Helyek:", len(profiles_df))
display(profiles_df.head())


## 3. Recommendation score oszlopok ellenőrzése

A recommendationhez a supporttal korrigált `*_adjusted` score-okat használjuk.


In [ ]:
DIMENSIONS = [
    "Food",
    "Drinks",
    "Service",
    "Ambience",
    "Price",
    "Location",
    "Overall",
]

ADJUSTED_COLS = {
    d: f"{d}_adjusted"
    for d in DIMENSIONS
}

missing = [
    col for col in ADJUSTED_COLS.values()
    if col not in profiles_df.columns
]

assert not missing, "Hiányzó oszlopok: " + ", ".join(missing)
print("Minden szükséges adjusted score elérhető.")


# 4. Hibrid preference parsing

A parser explicit-first logikát használ. Az egyértelmű kulcsszavas találatok és negációk elsőbbséget kapnak, a Sentence Transformer pedig csak a nem azonosított mondatrészeknél használatos szemantikus fallbackként.

In [ ]:
DIMENSION_DESCRIPTIONS = {
    "Food": (
        "quality, taste and variety of food, meals, dishes, cuisine and menu"
    ),
    "Drinks": (
        "quality and variety of coffee, drinks, wine, beer, cocktails and beverages"
    ),
    "Service": (
        "quality, speed and friendliness of service, staff, waiters and servers"
    ),
    "Ambience": (
        "atmosphere, ambience, vibe, coziness, comfort, noise level, interior and space"
    ),
    "Price": (
        "value for money, price satisfaction, affordability and whether prices feel reasonable"
    ),
    "Location": (
        "quality and convenience of location, neighborhood and area"
    ),
    "Overall": (
        "overall restaurant experience, general satisfaction and overall quality"
    ),
}

display(
    pd.DataFrame({
        "dimension": list(DIMENSION_DESCRIPTIONS.keys()),
        "description": list(DIMENSION_DESCRIPTIONS.values()),
    })
)


## 4.1. Sentence Transformer modell betöltése

In [ ]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=DEVICE,
)

_dimension_names = list(DIMENSION_DESCRIPTIONS.keys())
_dimension_texts = [DIMENSION_DESCRIPTIONS[d] for d in _dimension_names]

_dimension_embeddings = embedding_model.encode(
    _dimension_texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

print("Embedding shape:", _dimension_embeddings.shape)


## 4.2. Hibrid parser – szabályok és helper függvények

In [ ]:
VENUE_PATTERNS = {
    "Brunch/Breakfast/Bakery": [r"\bbrunch\b", r"\bbreakfast\b", r"\bbakery\b"],
    "Cafe/Coffee": [r"\bcafe\b", r"\bcafé\b", r"\bcoffee shop\b", r"\bcoffee place\b"],
    "Restaurant": [r"\brestaurant\b", r"\bdinner place\b", r"\bdinner spot\b"],
}

DIMENSION_KEYWORDS = {
    "Food": ["food", "meal", "meals", "dish", "dishes", "cuisine", "menu", "tasty", "delicious"],
    "Drinks": ["coffee", "drink", "drinks", "wine", "beer", "cocktail", "cocktails", "beverage", "beverages"],
    "Service": ["service", "staff", "waiter", "waitress", "server", "servers", "attentive", "hospitality"],
    "Ambience": ["ambience", "ambiance", "atmosphere", "vibe", "cozy", "cosy", "romantic", "quiet", "lively", "interior", "spacious", "noise", "noisy"],
    "Price": ["price", "prices", "value for money", "good value", "reasonable price", "reasonable prices", "worth the price", "worth it"],
    "Location": ["location", "neighborhood", "neighbourhood", "area", "central", "convenient location"],
    "Overall": ["overall", "overall experience", "general experience"],
}

NOT_IMPORTANT_PATTERNS = [
    "not important", "is not important", "isn't important", "don't care", "do not care",
    "doesn't matter", "does not matter", "not a priority",
]

STRONG_PREFERENCE_PATTERNS = [
    r"\bgreat\b", r"\bexcellent\b", r"\bamazing\b", r"\boutstanding\b", r"\bfantastic\b",
    r"\bexceptional\b", r"\bvery good\b", r"\breally good\b", r"\bvery important\b",
    r"\breally important\b", r"\bpriority\b", r"\bmust[- ]?have\b", r"\bessential\b",
]

BUDGET_PATTERNS = [r"\bcheap\b", r"\baffordable\b", r"\bbudget\b", r"\binexpensive\b", r"\blow[- ]cost\b"]


def split_preference_clauses(text):
    text = text.lower()
    clauses = re.split(r"[.!?;,]+|\bbut\b|\band\b|\bwith\b|\bwhile\b|\bplus\b", text)
    return [clause.strip() for clause in clauses if clause.strip()]


def detect_venue_type(text):
    text_lower = text.lower()
    for venue_type, patterns in VENUE_PATTERNS.items():
        if any(re.search(pattern, text_lower) for pattern in patterns):
            return venue_type
    return None


def contains_keyword(text, keyword):
    pattern = r"\b" + re.escape(keyword) + r"\b"
    return bool(re.search(pattern, text))


def detect_dimensions_in_clause(clause):
    matched = {}
    for dimension, keywords in DIMENSION_KEYWORDS.items():
        matches = [keyword for keyword in keywords if contains_keyword(clause, keyword)]
        if matches:
            matched[dimension] = matches
    return matched


def clause_is_not_important(clause):
    return any(pattern in clause for pattern in NOT_IMPORTANT_PATTERNS)


def clause_has_strong_preference(clause):
    return any(re.search(pattern, clause) for pattern in STRONG_PREFERENCE_PATTERNS)


def detect_budget_request(text):
    text_lower = text.lower()
    return any(re.search(pattern, text_lower) for pattern in BUDGET_PATTERNS)

## 4.3. Explicit-first parser + semantic fallback

A negáció felülír minden más szabályt. Explicit találatnál nincs szükség embedding-alapú döntésre. A semantic fallback csak olyan clause-ra fut, ahol nem találtunk explicit recommendation-dimenziót; az `Overall` nem lehet semantic fallback eredménye.

In [ ]:
SEMANTIC_THRESHOLD = 0.45
SEMANTIC_MARGIN = 0.05
SEMANTIC_WEIGHT = 0.75


def semantic_fallback(clause, ignored_dimensions):
    clause_embedding = embedding_model.encode(
        [clause],
        normalize_embeddings=True,
        convert_to_numpy=True,
    )[0]

    similarities = _dimension_embeddings @ clause_embedding
    candidates = []

    for dimension, similarity in zip(_dimension_names, similarities):
        if dimension == "Overall" or dimension in ignored_dimensions:
            continue
        candidates.append((dimension, float(similarity)))

    candidates = sorted(candidates, key=lambda x: x[1], reverse=True)
    if not candidates:
        return None

    best_dimension, best_similarity = candidates[0]
    second_similarity = candidates[1][1] if len(candidates) > 1 else 0.0
    margin = best_similarity - second_similarity

    if best_similarity >= SEMANTIC_THRESHOLD and margin >= SEMANTIC_MARGIN:
        return {
            "dimension": best_dimension,
            "similarity": best_similarity,
            "margin": margin,
        }
    return None


def parse_user_preference(text):
    clauses = split_preference_clauses(text)
    raw_weights = {dimension: 0.0 for dimension in DIMENSIONS}
    evidence = {dimension: [] for dimension in DIMENSIONS}
    ignored_dimensions = set()
    unmatched_clauses = []

    for clause in clauses:
        matched_dimensions = detect_dimensions_in_clause(clause)

        if not matched_dimensions:
            unmatched_clauses.append(clause)
            continue

        if clause_is_not_important(clause):
            for dimension in matched_dimensions:
                ignored_dimensions.add(dimension)
                raw_weights[dimension] = 0.0
                evidence[dimension].append({"type": "ignored", "clause": clause})
            continue

        weight = 1.5 if clause_has_strong_preference(clause) else 1.0
        for dimension, keywords in matched_dimensions.items():
            if dimension in ignored_dimensions:
                continue
            raw_weights[dimension] = max(raw_weights[dimension], weight)
            evidence[dimension].append({
                "type": "explicit",
                "clause": clause,
                "keywords": keywords,
                "weight": weight,
            })

    semantic_matches = []
    for clause in unmatched_clauses:
        if detect_venue_type(clause) is not None or len(clause.split()) < 2:
            continue
        result = semantic_fallback(clause, ignored_dimensions)
        if result is None:
            continue
        dimension = result["dimension"]
        if dimension in ignored_dimensions:
            continue
        raw_weights[dimension] = max(raw_weights[dimension], SEMANTIC_WEIGHT)
        evidence[dimension].append({
            "type": "semantic",
            "clause": clause,
            "similarity": result["similarity"],
            "margin": result["margin"],
            "weight": SEMANTIC_WEIGHT,
        })
        semantic_matches.append({"clause": clause, **result})

    for dimension in ignored_dimensions:
        raw_weights[dimension] = 0.0

    fallback_used = False
    total = sum(raw_weights.values())
    if total == 0:
        raw_weights["Overall"] = 1.0
        total = 1.0
        fallback_used = True

    weights = {dimension: raw_weights[dimension] / total for dimension in DIMENSIONS}

    return {
        "original_text": text,
        "clauses": clauses,
        "venue_type": detect_venue_type(text),
        "budget_request": detect_budget_request(text),
        "ignored_dimensions": sorted(ignored_dimensions),
        "raw_weights": raw_weights,
        "weights": weights,
        "evidence": evidence,
        "semantic_matches": semantic_matches,
        "fallback_used": fallback_used,
    }

## 4.4. Első preference parser teszt

In [ ]:
example_preference = (
    "I want a cozy brunch place with great food and good service. "
    "Price is not important."
)

parsed_example = parse_user_preference(example_preference)

print("Venue type:", parsed_example["venue_type"])
print("Ignored dimensions:", parsed_example["ignored_dimensions"])
print("Budget request:", parsed_example["budget_request"])
print("Clauses:", parsed_example["clauses"])

parser_result_df = pd.DataFrame({
    "dimension": DIMENSIONS,
    "raw_weight": [parsed_example["raw_weights"][d] for d in DIMENSIONS],
    "final_weight": [parsed_example["weights"][d] for d in DIMENSIONS],
})

display(parser_result_df.sort_values("final_weight", ascending=False))

for dimension in DIMENSIONS:
    if parsed_example["evidence"][dimension]:
        print(f"\n{dimension}:")
        for item in parsed_example["evidence"][dimension]:
            print(item)

# 5. Recommendation score

A végső score két részből áll:

### Preference match
A kiválasztott dimenziók `*_adjusted` score-jainak preference-súlyozott átlaga.

### Overall stabilizátor
Ha az `Overall` nincs explicit módon az aktív preference-dimenziók között, akkor kis, **10%-os** stabilizáló súlyt kap.

`final_score = 0.90 × preference_match + 0.10 × Overall_adjusted`

Ez csökkenti annak kockázatát, hogy egy hely egyetlen erős részdimenzió miatt kerüljön túl magasra.


In [ ]:
OVERALL_STABILIZER_WEIGHT = 0.10


def calculate_recommendations(
    preference_text,
    top_n=3,
    strict_venue_filter=True,
):
    parsed = parse_user_preference(preference_text)
    weights = parsed["weights"]
    venue_type = parsed["venue_type"]

    candidates = profiles_df.copy()

    if strict_venue_filter and venue_type is not None:
        filtered = candidates.loc[
            candidates["venue_type"] == venue_type
        ].copy()

        if len(filtered) > 0:
            candidates = filtered

    preference_score = np.zeros(len(candidates), dtype=float)

    for dimension in DIMENSIONS:
        weight = weights[dimension]
        if weight <= 0:
            continue

        values = (
            candidates[ADJUSTED_COLS[dimension]]
            .fillna(50)
            .to_numpy(dtype=float)
        )

        preference_score += weight * values

    candidates["preference_match"] = preference_score

    if weights["Overall"] == 0:
        overall = (
            candidates["Overall_adjusted"]
            .fillna(50)
            .to_numpy(dtype=float)
        )

        candidates["recommendation_score"] = (
            (1 - OVERALL_STABILIZER_WEIGHT)
            * candidates["preference_match"]
            + OVERALL_STABILIZER_WEIGHT
            * overall
        )
    else:
        candidates["recommendation_score"] = candidates["preference_match"]

    candidates = candidates.sort_values(
        "recommendation_score",
        ascending=False,
    )

    return candidates.head(top_n).copy(), parsed


# 6. Magyarázható Top-N output

Az indoklás közvetlenül a kiválasztott preference-dimenziókból és a place score-okból készül, nem generatív LLM-szöveg.


In [ ]:
def build_explanation(place_row, parsed, max_dimensions=3):
    weights = parsed["weights"]

    active = [
        d for d in DIMENSIONS
        if weights[d] > 0
    ]

    ranked = sorted(
        active,
        key=lambda d: weights[d] * float(place_row[ADJUSTED_COLS[d]]),
        reverse=True,
    )[:max_dimensions]

    parts = [
        f"{d}: {float(place_row[ADJUSTED_COLS[d]]):.1f}"
        for d in ranked
    ]

    return "Strongest preference matches – " + ", ".join(parts) + "."


def recommend(
    preference_text,
    top_n=3,
    strict_venue_filter=True,
):
    recommendations, parsed = calculate_recommendations(
        preference_text,
        top_n=top_n,
        strict_venue_filter=strict_venue_filter,
    )

    rows = []

    for rank, (_, row) in enumerate(recommendations.iterrows(), start=1):
        rows.append({
            "rank": rank,
            "name": row["name"],
            "venue_type": row["venue_type"],
            "recommendation_score": round(float(row["recommendation_score"]), 1),
            "explanation": build_explanation(row, parsed),
        })

    return pd.DataFrame(rows), parsed


## 6.1. Első teljes recommendation példa

In [ ]:
preference_1 = (
    "I want a cozy brunch place with great food and good service. "
    "Price is not important."
)

recommendations_1, parsed_1 = recommend(
    preference_1,
    top_n=3,
)

print("User preference:")
print(preference_1)

print("\nDetected venue type:")
print(parsed_1["venue_type"])

print("\nIgnored dimensions:")
print(parsed_1["ignored_dimensions"])

print("\nPreference weights:")
display(
    pd.DataFrame({
        "dimension": DIMENSIONS,
        "weight": [parsed_1["weights"][d] for d in DIMENSIONS],
    }).sort_values("weight", ascending=False)
)

print("\nTop recommendations:")
display(recommendations_1)


# 7. További tesztprofilok

A recommendation rendszernek több, eltérő preference-profilt is érdemes konzisztensen kezelnie.


In [ ]:
TEST_PROFILES = {
    "Brunch + food + ambience": (
        "I want a cozy brunch place with excellent food and a nice atmosphere."
    ),
    "Cafe + coffee + ambience": (
        "I'm looking for a cafe with great coffee and a cozy atmosphere."
    ),
    "Restaurant + service": (
        "I want a restaurant with excellent service and really good food."
    ),
    "Value focused": (
        "I care about good value for money and good food."
    ),
    "Location focused": (
        "I want a restaurant in a convenient location with a nice atmosphere."
    ),
}


## 7.0. Parser sanity checkek

In [ ]:
for profile_name, preference in TEST_PROFILES.items():
    parsed = parse_user_preference(preference)
    active_weights = {
        dimension: round(weight, 3)
        for dimension, weight in parsed["weights"].items()
        if weight > 0
    }
    print("=" * 70)
    print(profile_name)
    print("Venue:", parsed["venue_type"])
    print("Ignored:", parsed["ignored_dimensions"])
    print("Budget request:", parsed["budget_request"])
    print("Weights:", active_weights)

In [ ]:
all_test_results = []
all_test_preferences = []

for profile_name, preference in TEST_PROFILES.items():
    recs, parsed = recommend(preference, top_n=3)

    temp = recs.copy()
    temp.insert(0, "test_profile", profile_name)
    all_test_results.append(temp)

    for dimension in DIMENSIONS:
        all_test_preferences.append({
            "test_profile": profile_name,
            "dimension": dimension,
            "weight": parsed["weights"][dimension],
            "venue_type": parsed["venue_type"],
        })

test_results_df = pd.concat(all_test_results, ignore_index=True)
test_preference_weights_df = pd.DataFrame(all_test_preferences)

display(test_results_df)


## 7.1. Felismert preference-súlyok ellenőrzése

In [ ]:
display(
    test_preference_weights_df.pivot(
        index="test_profile",
        columns="dimension",
        values="weight",
    ).round(3)
)


# 8. Sanity checkek

Automatikusan ellenőrizzük, hogy:

- minden tesztprofilhoz 3 ajánlás készült;
- minden recommendation score 0–100 között van;
- a rangsor csökkenő score szerint rendezett;
- az explicit brunch/cafe/restaurant kéréseknél a Top-3 venue type megfelel a kérésnek.


In [ ]:
assert test_results_df["recommendation_score"].between(0, 100).all()

for profile_name in TEST_PROFILES:
    subset = test_results_df.loc[
        test_results_df["test_profile"] == profile_name
    ]

    assert len(subset) == 3

    scores = subset["recommendation_score"].tolist()
    assert scores == sorted(scores, reverse=True)

expected_types = {
    "Brunch + food + ambience": "Brunch/Breakfast/Bakery",
    "Cafe + coffee + ambience": "Cafe/Coffee",
    "Restaurant + service": "Restaurant",
    "Location focused": "Restaurant",
}

for profile_name, expected_type in expected_types.items():
    actual_types = set(
        test_results_df.loc[
            test_results_df["test_profile"] == profile_name,
            "venue_type",
        ]
    )

    assert actual_types == {expected_type}

print("Recommendation sanity checkek rendben.")


# 9. Top-3 vizualizáció

Az első példaprofil Top-3 recommendation score-ját ábrázoljuk.


In [ ]:
chart_df = recommendations_1.sort_values(
    "recommendation_score",
    ascending=True,
)

plt.figure(figsize=(8, 4))
plt.barh(
    chart_df["name"],
    chart_df["recommendation_score"],
)
plt.xlabel("Recommendation score")
plt.title("Top 3 personalized recommendations")
plt.xlim(0, 100)
plt.tight_layout()
plt.show()


# 10. Export

A tesztprofilok recommendation eredményeit és preference-súlyait külön CSV-be mentjük.


In [ ]:
RESULTS_PATH = Path("recommendation_test_results.csv")
WEIGHTS_PATH = Path("recommendation_test_preference_weights.csv")

test_results_df.to_csv(RESULTS_PATH, index=False)
test_preference_weights_df.to_csv(WEIGHTS_PATH, index=False)

print("Elmentve:")
print("-", RESULTS_PATH)
print("-", WEIGHTS_PATH)


## 11. Letöltés

In [ ]:
try:
    from google.colab import files
    files.download(str(RESULTS_PATH))
    files.download(str(WEIGHTS_PATH))
except ImportError:
    print("A CSV-k a notebook mappájában találhatók.")


# 12. Projekt pipeline – végső állapot

A teljes alkalmazási lánc:

**SemEval-2016 annotációk**  
→ Transformer Aspect Detection  
→ Transformer Aspect-Level Sentiment Classification  
→ Google Local review-k elemzése  
→ support-adjusted place profile-ok  
→ természetes nyelvű user preference  
→ Sentence Transformer semantic preference matching  
→ content-based ranking  
→ Top-N personalized recommendation

## A recommendation megközelítés jellemzői

- content-based;
- természetes nyelvű preference input;
- semantic embedding alapú dimenziófelismerés;
- explicit venue-type filtering;
- supporttal korrigált place profile-ok;
- átlátható, reprodukálható ranking;
- score-alapú magyarázható output.

## Fontos korlátok

1. A recommendation jelenleg 20 kiválasztott washingtoni helyből választ.
2. A preference-dimenziók száma előre rögzített.
3. A `Price` value/price satisfactiont mér, nem objektív árszintet.
4. Nincs explicit user–item ground-truth dataset, ezért a recommendation részt főként sanity checkekkel és kvalitatív tesztprofilokkal értékeljük.
5. A place profile-ok az upstream Aspect Detection és Sentiment Classification hibáit is öröklik.
6. A semantic preference parser külső pretrained embedding modellt használ, de azt nem fine-tuningoljuk ezen a projekten.

Ezek természetes továbbfejlesztési irányokat is jelentenek.
